<a href="https://colab.research.google.com/github/SULAIMAN-5-AHMED/FlyRankWeek1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SULAIMAN-5-AHMED/FlyRankWeek1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring. My lane is about measuring how well CTR, impressions, and position translate into engagement for a given piece of content — a continuous, per-row output (e.g., an engagement-signal score built from ctr, avg_position, and impressions_90d/impressions_last_30d, validated against engagement_rate/scroll_rate/engaged_sessions_90d) rather than a discrete label. Classification doesn't fit because there's no natural fixed category I'm predicting; clustering doesn't fit because I need output tied explicitly to a real engagement metric, not an unlabeled grouping; and ranking (learning-to-rank) doesn't fit either, since it optimizes relative order between items rather than an absolute measure of signal-to-engagement strength — worth noting since "ranking" in my lane's name refers to the search-ranking signals themselves, not the ML task type. Since the data has no calendar dates, this is best understood as a correlational/diagnostic scoring task — describing how these signals associate with engagement now, not forecasting future engagement.

In [1]:
!git clone https://github.com/SULAIMAN-5-AHMED/FlyRankWeek1.git

Cloning into 'FlyRankWeek1'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 144 (delta 54), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 1.89 MiB | 15.73 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("/content/FlyRankWeek1/data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: CTR (ctr column) — and it's an observed outcome, not a defined rule. It's already computed directly from real user behavior (clicks_90d / impressions_90d, verified against the raw columns with a max difference of 0.005), not something I'm inventing thresholds or scoring rules for. One honest caveat to include: CTR alone is a decent proxy for initial appeal (does the snippet/title get people to click), but it's not the full engagement story your lane is scoped to — engagement_rate, scroll_rate, and engaged_sessions_90d capture what happens after the click, which CTR can't see. So CTR is the primary label, with position (avg_position) and impressions (impressions_last_30d/impressions_prev_30d) as the signals being tested against it, and the post-click engagement fields held in reserve as a secondary check on whether high-CTR content is also genuinely engaging, not just clickbait-effective.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(df['content_id'].duplicated().sum())
print(df['content_id'].nunique(), len(df))
print(df['client_id'].nunique())
print((df['impressions_90d'] < df['impressions_last_30d'] + df['impressions_prev_30d']).sum())

print(df['content_age_days'].min(), df['content_age_days'].max())
print(df['days_since_last_update'].min(), df['days_since_last_update'].max())

print(df.isnull().sum()[df.isnull().sum() > 0])

computed_ctr = (df['clicks_90d'] / df['impressions_90d'] * 100)
print((computed_ctr - df['ctr']).abs().max())
print(df['content_type'].value_counts())
print(df['main_intent'].value_counts(dropna=False))

0
30000 30000
32
0
90 564
1 373
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64
0.005000000000000782
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64
main_intent
informational    17235
transactional     5733
commercial        4612
NaN               2374
navigational        46
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Spearman rank correlation between avg_position and ctr. This is defensible because it directly tests the core premise of my lane — that position (a ranking signal) predicts engagement (CTR) — without assuming a linear relationship, which position-CTR curves rarely follow (clicks drop off sharply after the top few positions, not steadily). A strong negative correlation (something like |r| ≥ 0.5, i.e. better position → meaningfully higher CTR) means "good" — the ranking signal is doing real explanatory work. A weak correlation (|r| closer to 0) would mean position alone doesn't explain much of the CTR variation, and I'd need impressions or content-level factors to carry more of the signal. I'd compute it directly: df[['avg_position','ctr']].corr(method='spearman'), and treat it as descriptive/diagnostic — a correlation this strong or weak, not a claim about causation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one piece of content (content_id), carrying its ranking signals (avg_position, impressions_*) alongside its engagement outcome (ctr, clicks_*, engagement_rate, scroll_rate). Shape is (30000, 14), and duplicated().sum() == 0 confirms no content appears twice — the grain holds inside this slice too.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_cols = [
    'content_id', 'client_id',
    'avg_position', 'position_tier',
    'impressions_90d', 'impressions_last_30d', 'impressions_prev_30d', 'impression_tier',
    'ctr', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d',
    'engagement_rate', 'scroll_rate', 'engaged_sessions_90d'
]

lane_df = df[lane_cols]

print(lane_df.shape)
print(lane_df['content_id'].duplicated().sum(), 'duplicate content_ids')  # confirms grain
lane_df.head()

(30000, 15)
0 duplicate content_ids


,content_id,client_id,avg_position,position_tier,impressions_90d,impressions_last_30d,impressions_prev_30d,impression_tier,ctr,clicks_90d,clicks_last_30d,clicks_prev_30d,engagement_rate,scroll_rate,engaged_sessions_90d
0,content_304f48230142,client_f369cb89fc,10.6,striking,3803,578,987,good,0.76,29,2,13,5.88,4.55,1
1,content_a1fb4e703a9e,client_4e07408562,20.3,page_3_5,15320,2501,5915,good,0.05,7,2,1,0.00,10.00,0
2,content_9aa793d4d895,client_7f2253d7e2,36.5,page_3_5,12581,2382,6089,good,0.09,11,1,3,0.00,28.57,0
3,content_331d6c4de07b,client_19581e27de,6.2,page_1,11751,3626,4206,good,0.49,58,22,17,1.28,3.45,1
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,page_3_5,19140,4211,6452,good,0.13,24,10,2,0.00,24.29,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "if avg_position ≤ 3, CTR is high" breaks down because the position→CTR relationship isn't a clean step function — it's a noisy, non-linear decay curve that varies by other factors already in this data: main_intent (transactional vs informational queries click very differently at the same position), content_type, and impression_tier (a position-3 result with 50 impressions behaves differently than one with 5,000). A rule-based cutoff also can't account for the two missing-data patterns I already found (main_intent missing in ~8% of rows, search_volume/competition missing in ~8%) without silently misclassifying those rows. And because I'm treating this as a scoring task rather than classification, there's no natural threshold to hard-code in the first place — I need something that outputs a continuous number capturing how much position and impressions explain CTR variation across 30,000 rows and 32 different clients, not a single if/else boundary that would only fit one slice of the data and break on the rest.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.